In [26]:
# Force the Python kernel back to the absolute root directory
# import os
# os.chdir('/content')

# print("Safely back in the root directory. Ready to clone!")

Safely back in the root directory. Ready to clone!


In [1]:
!git clone https://github.com/ayoosh-bhatta4/teeny-tiny-ships.git
!pip install -r teeny-tiny-ships/organizer/requirements.txt

Cloning into 'teeny-tiny-ships'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 54 (delta 12), reused 25 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 14.35 KiB | 7.17 MiB/s, done.
Resolving deltas: 100% (12/12), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 7.6 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

# 1. This magically finds your .env file and loads it into the background
load_dotenv()

# 2. Now we pull the specific keys we need out of the background
groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv("HF_TOKEN")

# (Optional: A quick print statement just to verify it worked without printing the whole key)
print(f"✅ Groq Key loaded: {bool(groq_api_key)}")
print(f"✅ HF Token loaded: {bool(hf_token)}")

login(token=hf_token)
print("Successfully logged into huggingface!")


Successfully logged into huggingface!


In [3]:
%cd /content/teeny-tiny-ships/organizer

/content/teeny-tiny-ships/organizer


In [5]:
import json
import pandas as pd
import time
from groq import Groq

client = Groq(api_key=groq_api_key)

# We will store all our generated items in this list
all_training_examples = []
total_batches = 10  # 10 batches of 100 = 1000 examples

data_gen_prompt = """
You are an expert synthetic data generator.
Generate 100 realistic, highly varied examples of personal finance items (with prices) and map them to standard budgeting categories.
Categories to use: Food, Transport, Utilities, Entertainment, Shopping, Health, Home.

CRITICAL RULES:
1. Output strictly valid JSON.
2. The JSON must have a single root key called "dataset" containing a list of objects.
3. Each object must have a "text" key (the user input) and a "label" key (the category).
4. VARY THE FORMATS HEAVILY: Use slang, extreme typos, weird punctuation ("UBER-45"), long descriptions ("paid plumber 300 to fix sink"), and missing amounts. Make it chaotic but realistic.

EXPECTED FORMAT:
{
  "dataset": [
    {"text": "swiggy 200", "label": "Food"}
  ]
}
"""

print(f"🚀 Starting the Synthetic Data Factory. Generating {total_batches * 100} examples...")

for i in range(total_batches):
    try:
        print(f"Generating batch {i+1}/{total_batches}...")

        # High temperature (0.9) forces the model to be creative and not repeat the same examples
        response = client.chat.completions.create(
            messages=[{"role": "system", "content": data_gen_prompt}],
            model="llama-3.3-70b-versatile",
            temperature=0.9,
            response_format={"type": "json_object"}
        )

        # Parse the JSON and add the list of items to our master list
        raw_json = json.loads(response.choices[0].message.content)
        all_training_examples.extend(raw_json['dataset'])

        # Pause for 2 seconds to avoid Groq rate limits (Too Many Requests error)
        time.sleep(2)

    except Exception as e:
        print(f"⚠️ Error on batch {i+1}: {e}")

# Convert the massive list into a DataFrame and save it
df = pd.DataFrame(all_training_examples)

# Drop any exact duplicates the LLM might have accidentally generated
df = df.drop_duplicates(subset=['text'])

df.to_csv("finance_training_data_1k.csv", index=False)

print(f"\n✅ Success! Saved {len(df)} unique training examples to finance_training_data_1k.csv")
print(df.head())

🚀 Starting the Synthetic Data Factory. Generating 1000 examples...
Generating batch 1/10...
Generating batch 2/10...
Generating batch 3/10...
Generating batch 4/10...
Generating batch 5/10...
Generating batch 6/10...
Generating batch 7/10...
Generating batch 8/10...
Generating batch 9/10...
Generating batch 10/10...

✅ Success! Saved 1068 unique training examples to finance_training_data_1k.csv
                                   text          label
0                            swiggy 200           Food
1                     uber ride to work      Transport
2                   elec bill paid 1200      Utilities
3  watchin movie @ cinema.. cost me 500  Entertainment
4                 shopped @ mall for 3k       Shopping


In [6]:
!pip install accelerate -U

In [7]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load the NEW 1k Dataset
print("Loading the 1k dataset...")
df = pd.read_csv("finance_training_data_1k.csv")

# Quick safety check: Drop any rows where the LLM might have hallucinated empty text or labels
df = df.dropna(subset=['text', 'label'])

# 2. Label Encoding
unique_labels = df['label'].unique()
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}
df['label'] = df['label'].map(label2id)

# 3. Tokenization
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

print("Tokenizing 1,000 examples (this takes a few seconds)...")
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
split_dataset = tokenized_dataset.train_test_split(test_size=0.2)

# 4. Initialize Model
print("Loading DistilBERT Brain...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id
)

# 5. Training Setup
training_args = TrainingArguments(
    output_dir="./teeny_finance_model_1k",
    learning_rate=3e-5,                 # Standard learning rate
    per_device_train_batch_size=16,     # We can process 16 at a time now
    per_device_eval_batch_size=16,
    num_train_epochs=4,                 # 📉 Back down to 4 epochs because we have enough data!
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = np.mean(predictions == labels)
    return {"accuracy": accuracy}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

# 6. EXECUTE
print("\n🚀 Starting Production Training Loop...")
trainer.train()

Loading the 1k dataset...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=216, training_loss=0.7700261142518785, metrics={'train_runtime': 184.5844, 'train_samples_per_second': 18.506, 'train_steps_per_second': 1.17, 'total_flos': 452557052903424.0, 'train_loss': 0.7700261142518785, 'epoch': 4.0})

In [8]:
# Save the model, tokenizer, and config to a single final folder
trainer.save_model("./final_tiny_model")

# Now load the pipeline from THIS new folder
from transformers import pipeline

expense_classifier = pipeline(
    task="text-classification",
    model="./final_tiny_model",
    tokenizer="distilbert-base-uncased"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [9]:
from transformers import pipeline

# 1. Create an inference pipeline
# We point it directly to the folder where your Trainer just saved the best weights
print("Loading your custom model into the pipeline...")
expense_classifier = pipeline(
    task="text-classification",
    model="./final_tiny_model",
    tokenizer="distilbert-base-uncased" # We still use the base tokenizer
)

# 2. Let's do a vibe check with some brand new, unseen data!
test_inputs = [
    "swiggy order for 500",
    "paid the electric bill 1500",
    "bought some new sneakers 3000",
    "uber to the airport 450",
    "doctor copay 50"
]

print("\n🤖 Custom Model Predictions:")
print("-" * 30)

for text in test_inputs:
    # Pass the text to your model
    prediction = expense_classifier(text)[0]

    # Extract the label and confidence score
    category = prediction['label']
    confidence = round(prediction['score'] * 100, 2)

    print(f"Input: '{text}'")
    print(f"Category: {category} ({confidence}% confident)\n")

Loading your custom model into the pipeline...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


🤖 Custom Model Predictions:
------------------------------
Input: 'swiggy order for 500'
Category: Food (86.93% confident)

Input: 'paid the electric bill 1500'
Category: Utilities (89.46% confident)

Input: 'bought some new sneakers 3000'
Category: Shopping (91.08% confident)

Input: 'uber to the airport 450'
Category: Transport (89.51% confident)

Input: 'doctor copay 50'
Category: Health (91.49% confident)



In [11]:
import re

def organizer_custom(in_str):
    # 1. The Smarter Pre-process: Split the string immediately AFTER any number
    # This regex looks for digits (\d+), optional spaces (\s*), and splits right after them using a lookbehind (?<=...)
    # We also split by common separators just in case: ',' or '|' or 'and'

    # First, let's normalize the string by replacing separators with a special token so we can split easily later
    normalized_str = re.sub(r'(?i)\band\b|\||,', ' [SPLIT] ', in_str)

    # Next, insert a split token after any number that is followed by a letter (meaning a new item is starting)
    # e.g., "laptop 1200 mouse" -> "laptop 1200 [SPLIT] mouse"
    normalized_str = re.sub(r'(\d+)\s+(?=[a-zA-Z])', r'\1 [SPLIT] ', normalized_str)

    # Now, split the string using our token
    raw_parts = [part.strip() for part in normalized_str.split('[SPLIT]') if part.strip()]

    final_totals = {}

    for part in raw_parts:
        amounts = re.findall(r'\d+', part)
        if amounts:
            amount = int(amounts[-1])
            clean_text = re.sub(r'\d+', '', part).strip()

            if clean_text:
                prediction = expense_classifier(clean_text)[0]
                cat = prediction['label']
                final_totals[cat] = final_totals.get(cat, 0) + amount

    return final_totals

# --- REGEX ONLY TESTING ---
# We will just print the raw_parts to see if our split logic works before running the full evals.
def test_regex_split(in_str):
    normalized_str = re.sub(r'(?i)\band\b|\||,', ' [SPLIT] ', in_str)
    normalized_str = re.sub(r'(\d+)\s+(?=[a-zA-Z])', r'\1 [SPLIT] ', normalized_str)
    parts = [part.strip() for part in normalized_str.split('[SPLIT]') if part.strip()]
    print(f"Input: '{in_str}'")
    print(f"Parts: {parts}\n")

print("--- REGEX SPLIT TESTS ---")
test_regex_split("laptop 1200 mouse keyboard 50")
test_regex_split("sNeAkErS 150 hOoDiE 60")
test_regex_split("hammer 15 nails 5 drill 80")
test_regex_split("flux_capacitor 5000 kyber_crystal 2000")

--- REGEX SPLIT TESTS ---
Input: 'laptop 1200 mouse keyboard 50'
Parts: ['laptop 1200', 'mouse keyboard 50']

Input: 'sNeAkErS 150 hOoDiE 60'
Parts: ['sNeAkErS 150', 'hOoDiE 60']

Input: 'hammer 15 nails 5 drill 80'
Parts: ['hammer 15', 'nails 5', 'drill 80']

Input: 'flux_capacitor 5000 kyber_crystal 2000'
Parts: ['flux_capacitor 5000', 'kyber_crystal 2000']



In [10]:
import json

test_cases = [
    {
        "test_name": "Generalization - Missing Amounts V2",
        "input": "laptop 1200 mouse keyboard 50",
        "expected_sum": 1250
    },
    {
        "test_name": "Generalization - Conversational Story",
        "input": "Paid the plumber 350 to fix the sink and bought a new faucet for 120.",
        "expected_sum": 470
    },
    {
        "test_name": "Generalization - Chaotic Formatting",
        "input": "   sNeAkErS    150   hOoDiE 60  ",
        "expected_sum": 210
    },
    {
        "test_name": "Generalization - Heavy Punctuation",
        "input": "[water_bill]: 45 | (internet_plan) - 60",
        "expected_sum": 105
    },
    {
        "test_name": "Stress Test - Sci-Fi Nonsense",
        "input": "flux_capacitor 5000 kyber_crystal 2000",
        "expected_sum": 7000
    },
    {
        "test_name": "Stress Test - High Volume Hardware",
        "input": "hammer 15 nails 5 drill 80 screws 10 tape 4 paint 30 brushes 12 tarp 8",
        "expected_sum": 164
    },
    {
        "test_name": "Generalization - Long Multi-Word Descriptions",
        "input": "electric bill for january 120 weekend getaway to the mountains 400 fancy dinner date 80",
        "expected_sum": 600
    },
    {
        "test_name": "Edge Case - Number-like Words (Tricky!)",
        "input": "four tickets to the game 200 one dozen donuts 15",
        "expected_sum": 215
    }
]

with open('test_cases.json', 'w') as f:
    json.dump(test_cases, f)

print("✅ test_cases.json recreated in your current directory!")

✅ test_cases.json recreated in your current directory!


In [12]:
def run_custom_evals(test_file_path):
    print(f"--- Starting Custom Model Evals ---")
    with open(test_file_path, 'r') as file:
        tests = json.load(file)

    passed = 0
    for test in tests:
        print(f"\nRunning Test: {test['test_name']}")

        # We process the input string
        # Note: If input has multiple items, we might need to split by 'and' or ','
        # For now, let's see how it handles your standard test cases!
        result = organizer_custom(test['input'])

        actual_sum = sum(result.values())

        if actual_sum == test['expected_sum']:
            print(f"✅ PASS: Got {actual_sum}")
            passed += 1
        else:
            print(f"❌ FAIL: Expected {test['expected_sum']}, got {actual_sum}")
            print(f"   Model thought '{test['input']}' was {list(result.keys())}")

    print(f"\n--- Evals Complete: {passed}/{len(tests)} Passed ---")

# Run it!
run_custom_evals('test_cases.json')

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


--- Starting Custom Model Evals ---

Running Test: Generalization - Missing Amounts V2
✅ PASS: Got 1250

Running Test: Generalization - Conversational Story
✅ PASS: Got 470

Running Test: Generalization - Chaotic Formatting
✅ PASS: Got 210

Running Test: Generalization - Heavy Punctuation
✅ PASS: Got 105

Running Test: Stress Test - Sci-Fi Nonsense
✅ PASS: Got 7000

Running Test: Stress Test - High Volume Hardware
✅ PASS: Got 164

Running Test: Generalization - Long Multi-Word Descriptions
✅ PASS: Got 600

Running Test: Edge Case - Number-like Words (Tricky!)
✅ PASS: Got 215

--- Evals Complete: 8/8 Passed ---


In [13]:
import pandas as pd
from datasets import Dataset

# 1. Choose a cool name for your dataset repository
dataset_repo_name = "Organizer_finance_cat_dataset_1k"  # You can change this!

print(f"Loading local CSV and pushing to Hub as '{dataset_repo_name}'...")

# 2. Load the CSV we generated earlier
df = pd.read_csv("finance_training_data_1k.csv")
hf_dataset = Dataset.from_pandas(df)

# 3. Push it directly to your Hugging Face profile
hf_dataset.push_to_hub(dataset_repo_name)

print(f"✅ Dataset successfully published! Check your Hugging Face profile.")

Loading local CSV and pushing to Hub as 'Organizer_finance_cat_dataset_1k'...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 17.5kB / 17.5kB            

✅ Dataset successfully published! Check your Hugging Face profile.


In [14]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. Choose a name for your model repository
model_repo_name = "Organizer_DistilliBERT_model_custom" # You can change this!

print(f"Loading finalized model from local disk...")

# 2. Load your custom weights and the tokenizer from the finished folder
model = AutoModelForSequenceClassification.from_pretrained("./final_tiny_model")
tokenizer = AutoTokenizer.from_pretrained("./final_tiny_model")

print(f"Pushing model and tokenizer to Hub as '{model_repo_name}'...")

# 3. Push both the model brain and the tokenizer dictionary to the cloud
model.push_to_hub(model_repo_name)
tokenizer.push_to_hub(model_repo_name)

print(f"✅ Model successfully published! Your custom AI is now live on the internet.")

Loading finalized model from local disk...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Pushing model and tokenizer to Hub as 'Organizer_DistilliBERT_model_custom'...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0ztv0iw/model.safetensors:   3%|2         | 6.91MB /  268MB            

README.md: 0.00B [00:00, ?B/s]

✅ Model successfully published! Your custom AI is now live on the internet.
